# exp39 / TrOCR-CTC trained on synthetic handwriting (10 human handwriting styles (DiffusionPen bigrams + font unigrams), fitted to HHD geometry, LLMGen (DictaGen) text, 25k random subset), tested on HHD — training seed 44

**Model (same as exp38):** the ViT encoder of `microsoft/trocr-base-handwritten` (768-dim, 12 layers) + **one linear classifier** over the
reading-order frames, trained with CTC. No augmentation, no frozen phase.

**Training data:** `cyttic/heb-human10bi-hhdgeom-llmgen-25k` — 10 styles x 2,500 lines = 25,000 synthetic lines: the **size-ablation 25k subset** of `cyttic/heb-human10bi-hhdgeom-llmgen-250k` (fitted to the geometry of real HHD train lines; per style a random permutation, seed 42, first lines taken, so the 1k / 5k / 10k / 25k / 50k / 100k subsets are nested; the same rows in the machine, unigram and bigram sets of this corpus). **Nothing of HHD is trained on.** HHD is used only for
validation (dev writers) and for the final test.

**Same training steps as the 50k run (no epoch selection).** The 50k run of this set trained 4 epochs x 50,000 lines; this 25k run trains **8 epochs** x 25,000 lines = the same number of training steps (warm-up 5% + linear decay over the run). Phase 1 is skipped; the HHD dev writers are not used.

**Final** — fresh model, the synthetic data, the fixed epoch count above. Then the **HHD test** set
   (1,455 lines, 109 writers) is scored **once**: greedy, and beam search with the fixed character 6-gram **LM1** (built from BYP + DictaGen text, no HHD text; the same LM for every
   experiment; alpha 2.0, beta 6.0, beams 4 and 12), CER with and without punctuation, WER.

**Colab:** Runtime → **L4 GPU**; Secrets → `HF_TOKEN` (write); **Run all**. Output: public HF repo `cyttic/exp39-ctc-human10bi-hhdgeom-llmgen-25k-seed44`.


### 1. Setup

In [ ]:
!pip install -q -U "transformers==5.15.0" datasets jiwer

In [ ]:
import os, io, re, json, math, time, random
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, jiwer
from PIL import Image, ImageOps
from datasets import load_dataset, concatenate_datasets
from transformers import VisionEncoderDecoderModel
from huggingface_hub import hf_hub_download, HfApi, login, whoami
try:
    from google.colab import userdata; login(userdata.get("HF_TOKEN"))
except Exception:
    from huggingface_hub import get_token   # outside Colab (e.g. the A5000 server): the token saved by `hf auth login`
    if get_token() is None:
        from huggingface_hub import notebook_login; notebook_login()
print("HF user:", whoami()["name"])
device = "cuda" if torch.cuda.is_available() else "cpu"
BF16 = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8   # Ampere+ (L4/A100): bf16; T4: fp16
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16
print(device, torch.cuda.get_device_name(0) if device == "cuda" else "", "| amp", AMP_DTYPE)

### 2. Config

In [ ]:
CFG = json.loads('{"enc": "microsoft/trocr-base-handwritten", "row_h": 64, "sub": 2, "batch": 32, "lr_enc": 5e-05, "lr_head": 0.0003, "warmup": 0.05, "aug": 0.0, "seed": 44, "max_epochs": 8, "patience": 2, "final_epochs": null}')
ENC_SRC, ROW_H, SUB, BATCH = CFG["enc"], CFG["row_h"], CFG["sub"], CFG["batch"]
LR_ENC, LR_HEAD, WARMUP_FRAC, SEED = CFG["lr_enc"], CFG["lr_head"], CFG["warmup"], CFG["seed"]
MAX_EPOCHS, PATIENCE = CFG["max_epochs"], CFG["patience"]   # phase 1: epoch cap + early-stopping patience (epochs)
FINAL_EPOCHS = CFG["final_epochs"]                           # phase 2: None = best epoch of phase 1; or set an int by hand
SELECT_EPOCHS = False                                        # SAME STEPS: phase 1 skipped
FINAL_EPOCHS = 8   # = 4 epochs (the 50k run of this set) x 50,000 lines / 25,000 lines: the same number of training steps as the 50k run
AUGMENT = CFG["aug"] > 0                                     # off: this is the plain baseline
OUT_REPO = "cyttic/exp39-ctc-human10bi-hhdgeom-llmgen-25k-seed44"   # public
SYN_ID = "cyttic/heb-human10bi-hhdgeom-llmgen-25k"   # the synthetic training set (10 styles x 2,500 lines)
LM_REPO, LM_FILE = "cyttic/exp38-char-lms", "exp38_char6_bygen.npz"   # LM1: character 6-gram LM, BYP + DictaGen text, no HHD text; the SAME LM for every experiment
LM_ALPHA, LM_BETA, LM_BEAMS = 2.0, 6.0, (4, 12)   # LM1 setting (chosen on the HHD dev writers, exp39/eval_lm_nohhd.py); beam 4 = the reported one
HHD_ID = "cyttic/trocr-hebrew-matan"
CHARSET_REPO = "cyttic/exp35-trocr-large-en-ctc-seqhead-allreal-synpunct"   # same 198 classes as exp32-37
SPLIT_SEED, TEST_FRAC, DEV_FRAC = 42, 0.20, 0.20             # FIXED: HHD dev/test writer split, identical to exp36/37
NUM_WORKERS, EVAL_BS = 8, 64
RUN_FINAL = True                                             # False = stop after the epoch selection
PUSH = True
SMOKE = False                                                # True = tiny run to test the pipeline end to end

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT = "/content/exp39" if os.path.isdir("/content") else os.path.abspath("exp39_run")
os.makedirs(OUT, exist_ok=True); print("OUT", OUT, "|", CFG)

### 3. Charset (the 198 classes of exp32-37; class 0 = CTC blank)

In [ ]:
CHARS = json.load(open(hf_hub_download(CHARSET_REPO, "charset.json")))["chars"]   # CHARS[k] = class k; CHARS[0] = blank
assert CHARS[0] == ""
C2I = {c: i for i, c in enumerate(CHARS) if i}
N_CLASSES = len(CHARS)
def encode(text): return [C2I[c] for c in text if c in C2I]
def greedy(lp, n):
    best = lp[:n].argmax(-1).tolist(); out, prev = [], 0
    for k in best:
        if k != prev and k != 0: out.append(CHARS[k])
        prev = k
    return "".join(out)
print("classes:", N_CLASSES)

### 4. Data: HHD split + tiling

In [ ]:
class Tiler:
    '''mirror (RTL->LTR) -> ROW_H px high -> wrapped into the 384x384 square in rows of 384 px (no overlap, no autocrop).
    Returns the image tensor + the number of 16 px columns that hold ink, in reading order.'''
    C, P = 384, 16
    def __init__(self, row_h):
        self.H = row_h; self.R = self.C // row_h; self.max_cols = self.R * (self.C // self.P)
    def __call__(self, image):
        image = ImageOps.mirror(image.convert("RGB")); w, h = image.size
        nw = max(1, round(w * self.H / h)); image = image.resize((nw, self.H), Image.LANCZOS)
        cont = Image.new("RGB", (self.C, self.C), (255, 255, 255)); arr = np.array(image)
        sx, r = 0, 0
        while sx < nw and r < self.R:
            seg = arr[:, sx:sx + self.C]; cont.paste(Image.fromarray(seg), (0, r * self.H)); sx += seg.shape[1]; r += 1
        used = min(nw, self.R * self.C)
        t = torch.tensor(np.array(cont), dtype=torch.float32).permute(2, 0, 1) / 255.0
        return (t - 0.5) / 0.5, min(self.max_cols, math.ceil(used / self.P))
tiler = Tiler(ROW_H)
print("rows", tiler.R, "| max columns", tiler.max_cols, "| max CTC steps", tiler.max_cols * SUB)

# ---- HHD: validation (dev writers) and test only; nothing of HHD is trained on. Split identical to exp36/37/38 ----
hhd = load_dataset(HHD_ID)
orig_test_w = set("hhd:" + w for w in hhd["test"]["writer"])
hhd_all = concatenate_datasets([hhd["train"], hhd["test"]])
hw = ["hhd:" + w for w in hhd_all["writer"]]
wr = sorted(set(hw)); random.Random(SPLIT_SEED).shuffle(wr)
n_te, n_dev = int(len(wr) * TEST_FRAC), int(len(wr) * DEV_FRAC)
test_w, dev_w = set(wr[:n_te]), set(wr[n_te:n_te + n_dev])
assert test_w == orig_test_w, "test writers differ from the original HHD test split"
assert not (dev_w & orig_test_w)
test = hhd["test"]
tw = ["hhd:" + w for w in hhd["train"]["writer"]]
dev = hhd["train"].select([i for i, w in enumerate(tw) if w in dev_w])

# ---- the synthetic training set (both phases train on all of it) ----
syn = load_dataset(SYN_ID, split="train")
train_full = train_sel = syn
import collections; _per = collections.Counter(syn["style"]); print(f"synthetic train {len(syn)} lines; lines per style: {dict(sorted(_per.items()))}"); assert len(syn) == 25000 and len(_per) == 10 and set(_per.values()) == {2500}
print(f"HHD dev {len(dev)} lines / {len(dev_w)} writers (phase 1 validation) | HHD test {len(test)} lines / {len(orig_test_w)} writers (scored once, phase 2)")
if SMOKE:
    train_full = train_sel = syn.select(range(64)); dev, test = dev.select(range(32)), test.select(range(32))
    MAX_EPOCHS = 2; BATCH = 4; FINAL_EPOCHS = 2; LM_BEAMS = (4,)

def collate(batch):
    px, cols, labels, texts = [], [], [], []
    for ex in batch:
        t, c = tiler(ex["image"]); px.append(t); cols.append(c); labels.append(encode(ex["text"])); texts.append(ex["text"])
    return torch.stack(px), torch.tensor(cols), labels, texts

def loader(split, shuffle, bs=None):
    return torch.utils.data.DataLoader(split, batch_size=bs or BATCH, shuffle=shuffle, num_workers=NUM_WORKERS,
                                       collate_fn=collate, drop_last=False, persistent_workers=False)

# how many training lines cannot be aligned (label longer than the available CTC steps)?
_chk = [(len(encode(ex["text"])), tiler(ex["image"])[1] * SUB) for ex in train_full.select(range(min(len(train_full), 400)))]
print(f"CTC-feasible lines (first {len(_chk)} train lines): {sum(1 for l, T in _chk if 0 < l <= T)}/{len(_chk)}")

### 5. Model: TrOCR-base ViT encoder → reading-order frames → ONE linear classifier

In [ ]:
class EncoderLinearCTC(nn.Module):
    '''ViT patch tokens (24x24 grid) -> reading-order frames (one per 16 px column, the ROW_H//16 patch rows of a text row
    concatenated) -> a single Linear giving SUB CTC steps per column. Nothing else.'''
    def __init__(self, vit, n_classes):
        super().__init__()
        self.vit = vit
        if getattr(self.vit, "pooler", None) is not None: self.vit.pooler = None
        self.Hd = vit.config.hidden_size; self.n_special = 2 if vit.config.model_type == "deit" else 1   # CLS (+ distillation)
        self.k = ROW_H // 16; self.n_classes = n_classes
        self.classifier = nn.Linear(self.k * self.Hd, SUB * n_classes)
        nn.init.trunc_normal_(self.classifier.weight, std=0.02); nn.init.zeros_(self.classifier.bias)
    def forward(self, px):
        tok = self.vit(pixel_values=px).last_hidden_state[:, self.n_special:]            # (B, 576, H)
        B, R = tok.shape[0], tiler.R
        g = tok.reshape(B, 24, 24, self.Hd)[:, :R * self.k]
        g = g.reshape(B, R, self.k, 24, self.Hd).permute(0, 1, 3, 2, 4).reshape(B, R * 24, self.k * self.Hd)   # (B, 144, k*H)
        y = self.classifier(g).reshape(B, R * 24 * SUB, self.n_classes)
        return F.log_softmax(y.float(), dim=-1)

def build_model():
    '''a FRESH model: TrOCR-base encoder weights + a newly initialised classifier (called once per phase)'''
    torch.manual_seed(SEED)
    vit = VisionEncoderDecoderModel.from_pretrained(ENC_SRC).encoder
    return EncoderLinearCTC(vit, N_CLASSES).to(device)

model = build_model()
n_enc = sum(p.numel() for p in model.vit.parameters()); n_head = sum(p.numel() for p in model.classifier.parameters())
print(f"encoder {n_enc/1e6:.1f}M ({model.vit.config.model_type}, hidden {model.vit.config.hidden_size}) | classifier {n_head/1e6:.3f}M")

### 6. Metrics + training function

In [ ]:
def scores(R, H):
    o = {}
    for v, f in {"raw": lambda t: t, "let": lambda t: re.sub(r" +", " ", re.sub(r"[^א-ת ]", " ", t)).strip()}.items():
        rr = [f(x) for x in R]; hh = [f(x) for x in H]; k = [i for i, x in enumerate(rr) if x]
        o[f"cer_{v}"] = jiwer.cer([rr[i] for i in k], [hh[i] for i in k]); o[f"wer_{v}"] = jiwer.wer([rr[i] for i in k], [hh[i] for i in k])
    return o

@torch.no_grad()
def predict(model, split):
    model.eval(); H, R = [], []
    for px, cols, _, texts in loader(split, False, EVAL_BS):
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=device == "cuda"):
            lp = model(px.to(device))
        lp = lp.float().cpu().numpy()
        H += [greedy(lp[i], int(cols[i]) * SUB) for i in range(len(texts))]; R += texts
    return R, H

def ctc(lp, cols, labels):
    return F.ctc_loss(lp.transpose(0, 1), torch.tensor(sum(labels, [])), cols * SUB,
                      torch.tensor([len(l) for l in labels]), blank=0, zero_infinity=True)

def train(model, train_split, epochs, tag, eval_split=None, patience=None, decay=True):
    '''Fine-tune for `epochs` epochs (warm-up 5%; then linear decay over the whole run, or constant if decay=False). With eval_split: score it after every epoch,
    early-stop after `patience` epochs without improvement and restore the best epoch. Returns a small summary dict.'''
    log = os.path.join(OUT, f"train_log_{tag}.tsv")
    open(log, "w").write("epoch\tstep\ttrain_loss\tdev_cer\tdev_wer\tsec\n")
    head_params, enc_params = list(model.classifier.parameters()), list(model.vit.parameters())
    opt = torch.optim.AdamW([{"params": enc_params, "lr": LR_ENC}, {"params": head_params, "lr": LR_HEAD}], weight_decay=0.01)
    base_lr = [g["lr"] for g in opt.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda" and AMP_DTYPE == torch.float16)
    total = epochs * math.ceil(len(train_split) / BATCH); warm = max(1, int(WARMUP_FRAC * total)); step = 0
    best = {"cer": 9.9, "epoch": 0, "state": None}; bad = 0; t0 = time.time(); curve = []; ep = 0
    print(f"[{tag}] {len(train_split)} train lines | {math.ceil(len(train_split) / BATCH)} steps/epoch | up to {epochs} epochs")
    for ep in range(1, epochs + 1):
        losses = []
        for px, cols, labels, _ in loader(train_split, True):
            keep = [i for i, l in enumerate(labels) if 0 < len(l) <= int(cols[i]) * SUB]
            if not keep: continue
            f = min(1.0, (step + 1) / warm) * (max(0.0, (total - step) / max(1, total - warm)) if decay else 1.0)   # decay=False: constant LR after warm-up
            for g, b in zip(opt.param_groups, base_lr): g["lr"] = b * f
            model.train()
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=device == "cuda"):
                lp = model(px[keep].to(device))
            loss = ctc(lp, cols[keep], [labels[i] for i in keep])
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
            losses.append(float(loss)); step += 1
        tl = float(np.mean(losses))
        if eval_split is None:
            open(log, "a").write(f"{ep}\t{step}\t{tl:.4f}\t\t\t{time.time()-t0:.0f}\n")
            print(f"[{tag} epoch {ep}/{epochs}] loss {tl:.3f} | {time.time()-t0:.0f}s", flush=True); continue
        R, H = predict(model, eval_split); s = scores(R, H); curve.append({"epoch": ep, "train_loss": tl, **s})
        open(log, "a").write(f"{ep}\t{step}\t{tl:.4f}\t{s['cer_raw']:.4f}\t{s['wer_raw']:.4f}\t{time.time()-t0:.0f}\n")
        print(f"[{tag} epoch {ep}/{epochs}] loss {tl:.3f} | dev CER {100*s['cer_raw']:.2f} WER {100*s['wer_raw']:.1f} | {time.time()-t0:.0f}s", flush=True)
        if s["cer_raw"] < best["cer"] - 1e-4:
            best.update(cer=s["cer_raw"], epoch=ep, state={k: v.detach().to("cpu").clone() for k, v in model.state_dict().items()}); bad = 0
        else:
            bad += 1
            if patience and bad >= patience: print(f"early stop: no dev improvement for {patience} epochs"); break
    if best["state"] is not None: model.load_state_dict(best["state"])
    return {"epochs_run": ep, "best_epoch": best["epoch"], "best_dev_cer": best["cer"], "curve": curve}
print("ready")

### 7. Phase 1 — select the number of epochs (synthetic training, constant LR; HHD dev writers scored every epoch; test never touched)

In [ ]:
if SELECT_EPOCHS:
    sel = train(model, train_sel, MAX_EPOCHS, "select", eval_split=dev, patience=PATIENCE, decay=False)
    R, H = predict(model, dev); sel["dev"] = scores(R, H)
    print("best epoch:", sel["best_epoch"], "| dev (best epoch):", {k: round(v, 4) for k, v in sel["dev"].items()})
    if sel["best_epoch"] >= sel["epochs_run"] - 1: print("WARNING: best epoch is at the end of training; consider a larger MAX_EPOCHS")
    BEST_EPOCH = sel["best_epoch"]
    json.dump({"config": CFG, "train_lines": len(train_sel), "dev_lines": len(dev), **sel}, open(os.path.join(OUT, "results_select.json"), "w"), indent=1)
else:
    assert FINAL_EPOCHS, "SELECT_EPOCHS = False needs FINAL_EPOCHS"
    sel = {"skipped": True, "best_epoch": FINAL_EPOCHS, "epochs_run": 0}; BEST_EPOCH = FINAL_EPOCHS
    json.dump({"config": CFG, "train_lines": len(train_sel), "dev_lines": len(dev), **sel}, open(os.path.join(OUT, "results_select.json"), "w"), indent=1)
    print("phase 1 skipped; final epochs =", BEST_EPOCH)
del model; torch.cuda.empty_cache()


### 8. Phase 2 — final run (fresh model, same synthetic data, best epoch count with LR decay; HHD test scored once)

In [ ]:
results = {"config": CFG, "select": sel}
if RUN_FINAL:
    n_final = FINAL_EPOCHS or BEST_EPOCH
    print("FINAL: re-initialising from TrOCR-base and training for", n_final, "epochs on", len(train_full), "lines")
    model = build_model()
    fin = train(model, train_full, n_final, "final")
    R, H = predict(model, test); test_scores = scores(R, H)
    results["final"] = {"epochs": n_final, "train_lines": len(train_full), "test": test_scores}
    print("HHD TEST (greedy, raw CER/WER and Hebrew-letters-only CER):", {k: round(v, 4) for k, v in test_scores.items()})
    with open(os.path.join(OUT, "test_predictions.tsv"), "w", encoding="utf-8") as fh:
        fh.write("ref\thyp\n"); [fh.write(f"{r}\t{h}\n") for r, h in zip(R, H)]
    for i in range(3): print("REF", R[i]); print("OUT", H[i])
json.dump(results, open(os.path.join(OUT, "results.json"), "w"), indent=1)

### 8b. HHD test with a character LM (fixed setting) — CER with and without punctuation, WER

In [ ]:
%%writefile charlm.py
# -*- coding: utf-8 -*-
"""Character n-gram language model with interpolated Kneser-Ney smoothing -- our own
implementation (training + querying), used for shallow fusion in CTC beam search.

Training is vectorised with numpy: every n-gram of order <= N is hashed into one int64 key
(base 256, one "digit" per character code), counted with np.unique, and turned into
interpolated-KN probabilities stored ARPA-style -- log P(w | h) for every kept n-gram and a log
back-off weight for every kept history. Querying walks from the longest context down:
    P(w | h) = P_n(w | h)                        if the n-gram (h, w) is stored
             = bow(h) * P(w | h[1:])             otherwise (bow = 1 if h is unseen)
Interpolated KN (Chen & Goodman 1999): the top order uses raw counts, lower orders use
continuation counts N1+(. g); one absolute discount per order, D = n1 / (n1 + 2 n2).

Lines are left-padded with N-1 <s> symbols, so the first characters of a line have context and
every lower-order n-gram has a left extension. Saved as .npz (plain arrays, no pickle).
"""

import functools
import math
import re
import unicodedata

import numpy as np

B = 256                      # key base: one byte per character code
DICT_MAX = 2_000_000         # tables up to this size become Python dicts (fast); larger stay arrays


class _Table:
    """key -> value lookup over sorted int64 keys: a dict for small tables, binary search else."""

    def __init__(self, keys, vals, as_dict):
        self.d = dict(zip(keys.tolist(), vals.tolist())) if as_dict else None
        self.k, self.v = keys, vals

    def get(self, key, default=None):
        if self.d is not None:
            return self.d.get(key, default)
        i = int(np.searchsorted(self.k, key))
        return float(self.v[i]) if i < len(self.k) and int(self.k[i]) == key else default

    def __len__(self):
        return len(self.k)
BOS = 255                    # <s> code; real characters are 1..len(alphabet)


# ---- text normalisation shared by training and decoding ------------------------------------
_MAP = str.maketrans({"״": '"', "׳": "'", "־": "-", "–": "-", "—": "-", "‒": "-", "―": "-",
                      "“": '"', "”": '"', "„": '"', "‘": "'", "’": "'", " ": " "})
_NIQQUD = re.compile(r"[֑-ׇ]")


def normalize(text: str) -> str:
    """NFC, Hebrew points and cantillation removed, gershayim/geresh/maqaf/typographic quotes and
    dashes mapped to ASCII, whitespace collapsed."""
    t = _NIQQUD.sub("", unicodedata.normalize("NFC", text)).translate(_MAP)
    return re.sub(r"\s+", " ", t).strip()


class CharKN:
    def __init__(self, alphabet: str, order: int, logp: list, bow: list):
        self.alphabet, self.N = alphabet, order
        self.code = {c: i + 1 for i, c in enumerate(alphabet)}
        # per order n (index n-1): sorted keys + values (compact); small orders also as dicts
        self._arr_p = [(np.asarray(k, np.int64), np.asarray(v, np.float32)) for k, v in logp]
        self._arr_b = [(np.asarray(k, np.int64), np.asarray(v, np.float32)) for k, v in bow]
        self.logp = [_Table(k, v, as_dict=len(k) <= DICT_MAX) for k, v in self._arr_p]
        self.bow = [_Table(k, v, as_dict=len(k) <= DICT_MAX) for k, v in self._arr_b]
        # never-seen character: the unigram level's share for unseen symbols, spread uniformly
        self.unk = (float(self._arr_b[0][1][0]) if len(self._arr_b[0][1]) else 0.0) - math.log(max(1, len(alphabet)))

    # ---- querying ------------------------------------------------------------------------
    def logprob(self, context: str, char: str) -> float:
        """ln P(char | context); `context` is the text so far (any length)."""
        return self._lp(context[-(self.N - 1):] if self.N > 1 else "", char)

    @functools.lru_cache(maxsize=200_000)
    def _lp(self, ctx: str, char: str) -> float:
        w = self.code.get(char)
        if w is None:
            return self.unk
        codes = [self.code.get(c, 0) for c in ctx]
        if 0 in codes:                                   # unknown char: context restarts after it
            codes = codes[len(codes) - codes[::-1].index(0):]
        codes = [BOS] * (self.N - 1 - len(codes)) + codes
        acc = 0.0
        for n in range(self.N, 0, -1):                   # longest context first
            h = codes[len(codes) - (n - 1):] if n > 1 else []
            hk = 0
            for c in h:
                hk = hk * B + c
            gk = hk * B + w
            lp = self.logp[n - 1].get(gk)
            if lp is not None:
                return acc + lp
            if n > 1:
                acc += self.bow[n - 1].get(hk, 0.0)
        return acc + self.unk                            # acc already holds every bow on the way down

    # ---- persistence ---------------------------------------------------------------------
    def save(self, path):
        arrs = {"alphabet": np.array(self.alphabet), "order": np.array(self.N)}
        for n in range(self.N):
            for name, (k, v) in (("p", self._arr_p[n]), ("b", self._arr_b[n])):
                arrs[f"{name}{n+1}_k"], arrs[f"{name}{n+1}_v"] = k, v
        np.savez_compressed(path, **arrs)

    @classmethod
    def load(cls, path):
        z = np.load(path, allow_pickle=False)
        N = int(z["order"])
        return cls(str(z["alphabet"]), N,
                   [(z[f"p{n}_k"], z[f"p{n}_v"]) for n in range(1, N + 1)],
                   [(z[f"b{n}_k"], z[f"b{n}_v"]) for n in range(1, N + 1)])


# ---- training ------------------------------------------------------------------------------
def _encode(lines, code, N):
    """-> one int16 array: each line = (N-1) x <s> + its codes, lines separated by 0."""
    parts = []
    pad = [BOS] * (N - 1)
    for ln in lines:
        parts.append(np.array(pad + [code[c] for c in ln] + [0], np.int64))
    return np.concatenate(parts) if parts else np.zeros(0, np.int64)


def _count(arr, n):
    """Unique n-gram keys (no separator inside, not ending in <s>) with their counts."""
    L = len(arr) - n + 1
    if L <= 0:
        return np.zeros(0, np.int64), np.zeros(0, np.int64)
    key = np.zeros(L, np.int64); ok = np.ones(L, bool)
    for j in range(n):
        seg = arr[j:j + L]
        key = key * B + seg; ok &= seg != 0
    ok &= arr[n - 1:n - 1 + L] != BOS                 # never predict <s>
    return np.unique(key[ok], return_counts=True)


def train(lines, alphabet: str, order: int = 6, prune_min=(1, 1, 1, 2, 2, 2), chunk_chars=8_000_000, log=print):
    code = {c: i + 1 for i, c in enumerate(alphabet)}
    assert len(alphabet) < BOS
    # 1. raw counts per order, chunked, merged
    raw = [None] * order
    buf, size = [], 0
    def flush():
        arr = _encode(buf, code, order)
        for n in range(1, order + 1):
            k, c = _count(arr, n)
            if raw[n - 1] is None:
                raw[n - 1] = (k, c)
            else:
                kk = np.concatenate([raw[n - 1][0], k]); cc = np.concatenate([raw[n - 1][1], c])
                u, inv = np.unique(kk, return_inverse=True)
                raw[n - 1] = (u, np.bincount(inv, weights=cc).astype(np.int64))
    for ln in lines:
        buf.append(ln); size += len(ln)
        if size >= chunk_chars:
            flush(); buf, size = [], 0; log(f"  counted ... {len(raw[-1][0]):,} distinct {order}-grams")
    if buf:
        flush()
    # 2. KN "counts": raw at the top order, continuation counts below
    a = [None] * order
    a[order - 1] = raw[order - 1]
    for n in range(order - 1, 0, -1):
        k_up = raw[n][0]                               # distinct (n+1)-grams
        suf = k_up % (B ** n)                           # drop their first character
        u, c = np.unique(suf, return_counts=True)
        a[n - 1] = (u, c.astype(np.int64))
    # 3. interpolated KN probabilities, lowest order first
    V = len(alphabet)
    logp, bow = [], []
    P_prev = None                                       # (sorted keys, probs) of order n-1
    for n in range(1, order + 1):
        k, c = a[n - 1]
        n1, n2 = int((c == 1).sum()), int((c == 2).sum())
        D = n1 / (n1 + 2 * n2) if n1 + 2 * n2 > 0 else 0.5
        hist = k // B
        hu, hinv = np.unique(hist, return_inverse=True)
        denom = np.bincount(hinv, weights=c)
        nfollow = np.bincount(hinv)
        bw = D * nfollow / denom                        # interpolation weight per history
        if n == 1:
            lower = np.full(len(k), 1.0 / V)
        else:
            idx = np.searchsorted(P_prev[0], k % (B ** (n - 1)))
            lower = P_prev[1][idx]
        p = np.maximum(c - D, 0) / denom[hinv] + bw[hinv] * lower
        P_prev = (k, p)
        keep = c >= prune_min[n - 1]
        logp.append((k[keep], np.log(p[keep]).astype(np.float32)))
        hk = hu if n > 1 else np.array([0], np.int64)
        bow.append((hk, np.log(bw).astype(np.float32) if n > 1 else np.log(bw[:1]).astype(np.float32)))
        log(f"  order {n}: D={D:.3f} | {len(k):,} n-grams ({int(keep.sum()):,} kept) | {len(hu):,} histories")
    return CharKN(alphabet, order, logp, bow)


In [ ]:
# ---- test-set decoding with a character LM: CTC prefix beam search (identical to exp38/eval_ctc_wordlm.py, word term off) ----
import sys, multiprocessing as mp, unicodedata
sys.path.insert(0, os.getcwd())
import charlm
NEG = -1e30
LM = charlm.CharKN.load(hf_hub_download(LM_REPO, LM_FILE))
def _lse(a, b):
    if a < b: a, b = b, a
    return a if b <= NEG else a + math.log1p(math.exp(b - a))
GG = {"chars": CHARS, "lm": LM, "beam": 12}
def beam_search(args):
    logprobs_, length, alpha, beta = args
    chars, lm = GG["chars"], GG["lm"]; beam, top_k, min_logp, blank = GG["beam"], 8, -12.0, 0
    lp = np.asarray(logprobs_[:length], np.float64); lm_cache = {}
    def fuse(prefix, c):
        key = (prefix[-12:], c); v = lm_cache.get(key)
        if v is None:
            v = alpha * lm.logprob(prefix, c) + beta; lm_cache[key] = v
        return v
    beams = {"": (0.0, NEG)}
    for t in range(lp.shape[0]):
        row = lp[t]; cand = np.argpartition(row, -top_k)[-top_k:]; cand = [int(k) for k in cand if row[k] > min_logp]; nxt = {}
        for prefix, (pb, pnb) in beams.items():
            ptot = _lse(pb, pnb)
            for k in cand:
                p = row[k]
                if k == blank:
                    b, nb = nxt.get(prefix, (NEG, NEG)); nxt[prefix] = (_lse(b, ptot + p), nb); continue
                c = chars[k]
                if prefix and c == prefix[-1]:
                    b, nb = nxt.get(prefix, (NEG, NEG)); nxt[prefix] = (b, _lse(nb, pnb + p)); src = pb
                else:
                    src = ptot
                if src <= NEG: continue
                ext = prefix + c; b, nb = nxt.get(ext, (NEG, NEG)); nxt[ext] = (b, _lse(nb, src + p + fuse(prefix, c)))
        beams = dict(sorted(nxt.items(), key=lambda kv: _lse(*kv[1]), reverse=True)[:beam]) or {"": (0.0, NEG)}
    return max(beams.items(), key=lambda kv: _lse(*kv[1]))[0]

def scores_full(R, H):
    """CER with punctuation (raw), CER without punctuation (every Unicode punctuation character removed from reference AND hypothesis),
    WER on the raw text. Lines whose reference is empty are skipped."""
    nop = lambda t: re.sub(r" +", " ", "".join(c for c in t if not unicodedata.category(c).startswith("P"))).strip()
    def cer_of(f):
        k = [i for i in range(len(R)) if f(R[i]).strip()]
        return jiwer.cer([f(R[i]) for i in k], [f(H[i]) for i in k])
    k = [i for i, r in enumerate(R) if r.strip()]
    return {"cer": jiwer.cer([R[i] for i in k], [H[i] for i in k]), "wer": jiwer.wer([R[i] for i in k], [H[i] for i in k]), "cer_nopunct": cer_of(nop)}

@torch.no_grad()
def test_logprobs(model, split):
    model.eval(); out, R = [], []
    for px, cols, _, texts in loader(split, False, EVAL_BS):
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=device == "cuda"):
            lp = model(px.to(device))
        lp = lp.float().cpu().numpy(); out += [(lp[i], int(cols[i]) * SUB) for i in range(len(texts))]; R += texts
    return out, R

if RUN_FINAL:
    L, R = test_logprobs(model, test)
    final_decoding = {"greedy (no LM)": scores_full(R, [greedy(lp, n) for lp, n in L])}
    for bw in LM_BEAMS:
        GG["beam"] = bw
        with mp.get_context("fork").Pool(os.cpu_count()) as pool:
            H = pool.map(beam_search, [(lp, n, LM_ALPHA, LM_BETA) for lp, n in L], chunksize=8)
        final_decoding[f"beam {bw} + LM (alpha {LM_ALPHA}, beta {LM_BETA})"] = scores_full(R, H)
        with open(os.path.join(OUT, f"test_predictions_beam{bw}.tsv"), "w", encoding="utf-8") as fh:
            fh.write("ref\thyp\n"); [fh.write(f"{r}\t{h}\n") for r, h in zip(R, H)]
    results["final"]["test_decoding"] = final_decoding
    print("HHD TEST", "(LM = " + LM_REPO + "/" + LM_FILE + ")")
    for name, s in final_decoding.items():
        print(f"  {name:48s} CER {100*s['cer']:.2f} | CER w/o punct {100*s['cer_nopunct']:.2f} | WER {100*s['wer']:.2f}")
    json.dump(results, open(os.path.join(OUT, "results.json"), "w"), indent=1)


### 9. Save to the public HF repo

In [ ]:
from safetensors.torch import save_file
json.dump({"chars": CHARS}, open(os.path.join(OUT, "charset.json"), "w"), ensure_ascii=False)
if RUN_FINAL:   # weights of the FINAL model (full train set, last epoch); the select-phase model is not kept
    save_file({k: v.contiguous() for k, v in model.state_dict().items()}, os.path.join(OUT, "model.safetensors"))
if PUSH and not SMOKE:
    api = HfApi(); api.create_repo(OUT_REPO, private=False, exist_ok=True)
    api.upload_folder(repo_id=OUT_REPO, folder_path=OUT, commit_message="exp39 TrOCR-CTC on human10bi_hhdgeom_llmgen_25k_seed44, HHD test")
    print("pushed -> https://huggingface.co/" + OUT_REPO)
else:
    print("not pushed (SMOKE or PUSH=False); files in", OUT)